# Jev / TypeSafe System One 全用例实验

在一张支付宝首页截图（`alipay_small.png`，725×1568）上覆盖 Jev 的全部用例：

| 章节 | 内容 |
| --- | --- |
| §1 | 文本 state 上的三个决策原语：`Noul` / `Choice` / `Score` |
| §2 | 图片 state 上的五个原语：`Noul` / `Choice` / `Score` / `Box` / `Ground`（含像素/点击点换算、裁剪落盘） |
| §3 | 组合用例：criteria 嵌示例图、Noul criteria 嵌图、多图 state（历史帧）、单请求多问题、Box 低置信度处理 |
| §4 | 原生 HTTP：`assets` map + 指令字符串内联引用 |
| §5 | 错误用例（漏发图片头 → 400）、模型列表、用量统计 |

**前置条件**（均已验证）：
- 本 notebook 的内核是项目 `.venv`（已装 `typesafe_sdk`、`Pillow`、`requests`）
- `api.typesafe.ai` 在本区域被 451 封锁，需经本地代理 `http://localhost:7897` 出访
- 图片受 API 限制：PNG/JPEG、每边 256–1568 px、总像素 78,400–2,458,624；超限直接拒绝、不缩放
- `jev-latest` 当前仍是 1.13.0（实测 2026-10-15）；**Box 问题必须显式 `model="jev-preview"`**（1.14）
- 任何带图或带 Box 的请求都要发 `TypeSafe-Danger-Zone-Images: true` 头
- Ground 由 `gui-actor-3b`（GUI-Actor pointer head）回答：**只答 ground**，不能和文本原语混在一个请求；SDK 暂无 `Ground` 类，问题用原始 dict 发送、答案用 `response_model` 解析

> 安全提醒：API key 不要提交进 git；建议改为从环境变量或未跟踪的 `.env` 读取。

In [31]:
import os
from contextlib import contextmanager
from pathlib import Path

# 代理：api.typesafe.ai 在本区域 451，经本地代理出访（不用代理就删掉这两行）
# os.environ.setdefault("HTTP_PROXY", "http://localhost:7897")
# os.environ.setdefault("HTTPS_PROXY", "http://localhost:7897")

# # TODO: 换成从环境变量读取，不要把 key 提交进 git
os.environ.setdefault(
    "TYPESAFE_API_KEY",
    "apikey_2269ab1a92db19d4cf2b691003f56b1945d_b74cbe119cadbef8a20233d058c20e486991578f4e4d7893db4d26455f1528d9",
)
os.environ.setdefault("TYPESAFE_BASE_URL", "http://localhost:8008")

from typesafe_sdk import Box, Choice, Image, Noul, Score, TypeSafeClient

IMG_HEADERS = {"TypeSafe-Danger-Zone-Images": "true"}  # 带图/Box 请求必发
MODEL_PREVIEW = "jev-preview"  # Box 需要 1.14；不传 model 时默认 jev-latest（当前=1.13）

ROOT = Path("../assets")
SCREEN = ROOT / "alipay_small.png"
OUT = ROOT / "nb_out"
OUT.mkdir(exist_ok=True)


@contextmanager
def client():
    """每次调用独立建连，cell 之间互不影响。"""
    with TypeSafeClient() as c:
        yield c


def show(qid, ans):
    """按答案类型打印五种原语的返回。"""
    if ans.type == "noul":
        print(f"[{qid}] noul = {ans.noul:.3f}")
    elif ans.type == "choice":
        print(f"[{qid}] choice = {ans.choice!r}  confidence = {ans.confidence:.3f}")
        print(f"        probabilities = { {k: round(v, 3) for k, v in ans.probabilities.items()} }")
    elif ans.type == "score":
        print(f"[{qid}] score = {ans.score:.2f}  confidence = {ans.confidence:.3f}")
        print(f"        probabilities = { {k: round(v, 3) for k, v in ans.probabilities.items()} }")
    elif ans.type == "box":
        b = ans.box
        print(
            f"[{qid}] box: top={b.top:.3f} bottom={b.bottom:.3f} "
            f"left={b.left:.3f} right={b.right:.3f}  confidence = {ans.confidence:.3f}"
        )
    elif ans.type == "ground":
        print(f"[{qid}] point = {ans.point} (0-999)  confidence = {ans.confidence:.3f}")
        if ans.topk_points:
            print(f"        topk_points = {ans.topk_points}")
    else:
        print(f"[{qid}] {ans}")


def check_image(path):
    """打印图片尺寸并判断是否满足 API 限制，返回 (w, h)。"""
    from PIL import Image as PILImage

    w, h = PILImage.open(path).size
    px = w * h
    ok = (256 <= w <= 1568) and (256 <= h <= 1568) and (78400 <= px <= 2458624)
    status = "OK" if ok else "REJECTED（每边须 256-1568px，总像素 78400-2458624）"
    print(f"{Path(path).name}: {w}x{h}（{px} px）-> {status}")
    return w, h


print("setup ok；kernel 使用 .venv，SDK 已导入")

setup ok；kernel 使用 .venv，SDK 已导入


## §1 文本 state：三个决策原语

- `Noul`：判断条件是否成立，返回 yes 的概率（没有单独 confidence）
- `Choice`：从一组候选里选一个，返回分布与置信度
- `Score`：在有序等级上给出概率加权位置，用于分级/排序

In [32]:
with client() as c:
    r = c.system_one(
        state="The screen shows the Alipay home page with a search bar and a grid of shortcut icons.",
        questions={
            "is_payment_home": Noul(
                instructions="Is this the home screen of a payment app?"
            )
        },
    )
show("is_payment_home", r.answers["is_payment_home"])
print("model:", r.model, "| usage:", r.usage)

[is_payment_home] noul = 0.953
model: jev-latest | usage: input_tokens=167 output_tokens=0


In [33]:
# Choice：给一个目标描述，从已装应用里选应该打开的那个
with client() as c:
    r = c.system_one(
        state={
            "goal": "Send 20 yuan to Zhang San with the remark 'lunch'.",
            "installed_apps": ["Settings", "Camera", "Alipay", "WeChat", "Contacts", "Chrome"],
        },
        questions={
            "app_to_open": Choice(
                instructions="Which app in `installed_apps` should open to accomplish `goal`?",
                criteria={
                    "Alipay": "A payment app that can transfer money with a remark.",
                    "WeChat": "A messaging app that also supports transfers.",
                    "Contacts": "Only stores phone contacts; cannot pay.",
                    "none": "No listed app can accomplish the goal.",
                },
            )
        },
    )
show("app_to_open", r.answers["app_to_open"])

[app_to_open] choice = 'Alipay'  confidence = 0.698
        probabilities = {'Alipay': 0.698, 'WeChat': 0.275, 'Contacts': 0.011, 'none': 0.015}


In [34]:
# Score：有序等级上的分级（等级描述必须自成一体、可比较）
with client() as c:
    r = c.system_one(
        state={
            "message": "Your package will arrive today between 18:00 and 19:00. "
            "Please make sure someone is home to receive it."
        },
        questions={
            "urgency": Score(
                instructions="How urgent is `message` for the recipient?",
                criteria=[
                    "Informational only; no action needed.",
                    "Might need a small adjustment to plans.",
                    "Requires action within hours.",
                    "Requires immediate action.",
                ],
            )
        },
    )
show("urgency", r.answers["urgency"])
print("legend:", r.answers["urgency"].legend)

[urgency] score = 1.90  confidence = 0.823
        probabilities = {0: 0.037, 1: 0.081, 2: 0.823, 3: 0.059}
legend: {0: 'Informational only; no action needed.', 1: 'Might need a small adjustment to plans.', 2: 'Requires action within hours.', 3: 'Requires immediate action.'}


## §2 图片用例

- 图片放进 `state` 即成为 state 的一部分，所有问题共同可见；请求必带 `TypeSafe-Danger-Zone-Images: true`
- `Image(Path(...))` 由 SDK 自动 base64 上传并生成引用；同一个 `Image` 对象只加载一次、可复用
- **限制**：每边 256–1568 px、总像素 78,400–2,458,624；裁剪出来太小（如底部 tab bar 只裁 146px 高）会被整单拒绝
- Box 需要 1.14：`model="jev-preview"`
- `Ground`：把目标定位成一个**点击点**，`[x, y]` 归一化到 0–999 整数（Qwen-VL grounding 约定，左上原点），带置信度与 `topk_points` 备选区域中心；本机由 `gui-actor-3b` 回答，只答 ground，不能和上面的原语混在一个请求（见 §2.5）


In [35]:
# 校验主图尺寸，并裁两张后面要用的示例图（裁剪也要满足 256px 下限）
from PIL import Image as PILImage

w, h = check_image(SCREEN)

im = PILImage.open(SCREEN)
GRID = OUT / "crop_grid.png"      # 顶部图标区
TABBAR = OUT / "crop_tabbar.png"  # 底部 tab bar（裁高一点，避免低于 256px）
im.crop((0, int(h * 0.05), w, int(h * 0.30))).save(GRID)
im.crop((0, int(h * 0.78), w, h)).save(TABBAR)
check_image(GRID)
check_image(TABBAR)

alipay_small.png: 725x1568（1136800 px）-> OK
crop_grid.png: 725x392（284200 px）-> OK
crop_tabbar.png: 725x345（250125 px）-> OK


(725, 345)

In [36]:
# Noul on image：屏幕上是否存在某元素（存在性判断，Box 没有“未找到”答案，需搭配 Noul）
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "has_pay_button": Noul(
                instructions="Is there a 收付款 (pay/receive) button in `screen`?"
            ),
            "has_unread_badge": Noul(
                instructions="Is there a red unread badge anywhere in `screen`?"
            ),
        },
        extra_headers=IMG_HEADERS,
    )
for qid, ans in r.answers.items():
    show(qid, ans)
print("model:", r.model, "| usage:", r.usage, "（图片按 input token 计费，含在 input_tokens 里）")

[has_pay_button] noul = 0.967
[has_unread_badge] noul = 0.939
model: jev-preview | usage: input_tokens=1414 output_tokens=0 （图片按 input token 计费，含在 input_tokens 里）


In [37]:
# Choice on image：当前选中的底部 tab / 屏幕主体内容
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "selected_tab": Choice(
                instructions="Which bottom tab is currently selected in `screen`?",
                criteria={
                    "home": "首页 (Home) tab is highlighted.",
                    "life": "生活 (Life) tab is highlighted.",
                    "message": "消息 (Messages) tab is highlighted.",
                    "me": "我的 (Me) tab is highlighted.",
                },
            )
        },
        extra_headers=IMG_HEADERS,
    )
show("selected_tab", r.answers["selected_tab"])

[selected_tab] choice = 'home'  confidence = 0.977
        probabilities = {'home': 0.977, 'life': 0.01, 'message': 0.006, 'me': 0.007}


In [38]:
# Score on image：沿描述维度分级（这里看屏幕内容密度）
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "density": Score(
                instructions="How densely is `screen` filled with content?",
                criteria=[
                    "Mostly empty space.",
                    "About half of the screen is filled.",
                    "Densely packed with content.",
                ],
            )
        },
        extra_headers=IMG_HEADERS,
    )
show("density", r.answers["density"])

[density] score = 1.68  confidence = 0.724
        probabilities = {0: 0.046, 1: 0.23, 2: 0.724}


In [ ]:
# Box on image：定位「收付款」按钮，分数边换算成像素坐标，裁剪并画框落盘
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,  # Box 需要 1.14
        state={"screen": Image(SCREEN)},
        questions={
            "pay_button": Box(instructions="the 收付款 button in `screen`")
        },
        extra_headers=IMG_HEADERS,
    )

ans = r.answers["pay_button"]
show("pay_button", ans)

w, h = check_image(SCREEN)
b = ans.box
x0, y0 = round(b.left * w), round(b.top * h)
x1, y1 = round(b.right * w), round(b.bottom * h)
print(f"pixel box: ({x0}, {y0}) - ({x1}, {y1})；点击中心 ({(x0 + x1) // 2}, {(y0 + y1) // 2})")

im = PILImage.open(SCREEN)
im.crop((x0, y0, x1, y1)).save(OUT / "pay_button.png")
from PIL import ImageDraw

ImageDraw.Draw(im).rectangle([x0, y0, x1, y1], outline="red", width=4)
im.save(OUT / "pay_button_marked.png")
print("saved:", OUT / "pay_button.png", "和", OUT / "pay_button_marked.png")
print("model:", r.model, "| usage:", r.usage)

## §2.5 Ground：点击点定位

- `Ground` 把指令指向的目标定位成一个**点击点**：`[x, y]` 归一化到 0–999 整数（Qwen-VL grounding 约定：左上为原点，x 横向右、y 纵向下），附区域置信度；`topk_points` 是备选区域的中心，同尺度
- 与 Box 的分工：Box 给四边（适合裁剪、画框），Ground 给一个点（适合直接点击）；两者都没有“未找到”答案，存在性判断仍要搭配 Noul
- 本机由 `gui-actor-3b`（GUI-Actor pointer head）回答：**只答 ground**，和 Noul/Choice/Score 混在同一请求会 501；每个不同指令各跑一次 grounding pass，相同指令共享一次
- SDK 暂未提供 `Ground` 问题类：问题用原始 dict `{"type": "ground", ...}` 发送；返回的 `ground` 答案类型 SDK 也不认识（会直接丢弃），因此用 `response_model` 传自定义 Pydantic 模型解析
- `state` 对 GUI-Actor 是 system prompt：传空串时服务器回退到 checkpoint 自带的 grounding system prompt（推荐；图片引用串不要留在 state 里）


In [39]:
# Ground on image：定位「收付款」按钮的点击点，0-999 归一化坐标换算成像素，画准星落盘
from pydantic import BaseModel


class GroundAnswer(BaseModel):
    """SDK 暂不认识的 ground 答案：point 为 0-999 整数点击点，topk_points 为备选区域中心。"""
    type: str
    point: tuple[int, int]
    confidence: float
    topk_points: list[tuple[int, int]] | None = None


class GroundResponse(BaseModel):
    model: str
    answers: dict[str, GroundAnswer]
    usage: dict | None = None


MODEL_GROUND = "gui-actor-3b"  # Ground 由 GUI-Actor pointer head 回答；不能和文本原语混在一个请求

with client() as c:
    r = c.system_one(
        model=MODEL_GROUND,
        state="",  # 空 state -> 服务器回退到 checkpoint 自带的 grounding system prompt
        assets={"screen": Image(SCREEN)},  # 图片经 assets 上传；grounding 落在（最后）一张图上
        questions={
            "pay_button": {"type": "ground", "instructions": "the 收付款 (pay/receive) button"},
        },
        extra_headers=IMG_HEADERS,
        response_model=GroundResponse,  # SDK 会丢弃 ground 类型答案，必须自定义解析
    )
show("pay_button", r.answers["pay_button"])

w, h = check_image(SCREEN)
gx, gy = r.answers["pay_button"].point
px, py = round(gx / 999 * w), round(gy / 999 * h)  # 0-999 -> 像素（左上原点）
print(f"click point: ({px}, {py}) px  <-  ({gx}, {gy}) / 999")

from PIL import ImageDraw

im = PILImage.open(SCREEN)
draw = ImageDraw.Draw(im)
rad = 28  # 准星半径
draw.ellipse([px - rad, py - rad, px + rad, py + rad], outline="red", width=4)
draw.line([px - rad - 12, py, px + rad + 12, py], fill="red", width=3)
draw.line([px, py - rad - 12, px, py + rad + 12], fill="red", width=3)
im.save(OUT / "pay_button_ground.png")
print("saved:", OUT / "pay_button_ground.png")
print("model:", r.model, "| usage:", r.usage)


[pay_button] point = (400, 150) (0-999)  confidence = 0.813
        topk_points = [(400, 150)]
alipay_small.png: 725x1568（1136800 px）-> OK
click point: (290, 235) px  <-  (400, 150) / 999
saved: ../assets/nb_out/pay_button_ground.png
model: gui-actor-3b | usage: {'input_tokens': 1593, 'output_tokens': 0}


In [40]:
# 一次请求多个 ground 问题：不同指令各跑一次 pass；低置信度时参考 topk_points 备选
with client() as c:
    r = c.system_one(
        model=MODEL_GROUND,
        state="",
        assets={"screen": Image(SCREEN)},
        questions={
            "pay_button": {"type": "ground", "instructions": "the 收付款 (pay/receive) button"},
            "scan_button": {"type": "ground", "instructions": "the 扫一扫 (scan) button"},
        },
        extra_headers=IMG_HEADERS,
        response_model=GroundResponse,
    )
for qid, ans in r.answers.items():
    show(qid, ans)

im = PILImage.open(SCREEN)
draw = ImageDraw.Draw(im)
for qid, ans in r.answers.items():
    gx, gy = ans.point
    px, py = round(gx / 999 * w), round(gy / 999 * h)
    color = "red" if ans.confidence >= 0.7 else "orange"  # 低置信度用橙色，并参考 topk_points
    draw.ellipse([px - 16, py - 16, px + 16, py + 16], outline=color, width=4)
    for tx, ty in ans.topk_points or []:
        qx, qy = round(tx / 999 * w), round(ty / 999 * h)
        draw.ellipse([qx - 8, qy - 8, qx + 8, qy + 8], outline=color, width=2)
im.save(OUT / "ground_multi.png")
print("saved:", OUT / "ground_multi.png")
print("model:", r.model, "| usage:", r.usage, "（每个不同指令一次 grounding pass，input_tokens 随之增加）")

# 混合原语会被拒：gui_actor 只答 ground，和 Noul 同请求 -> 501
from typesafe_sdk import TypeSafeInternalServerError

try:
    with client() as c:
        c.system_one(
            model=MODEL_GROUND,
            state="",
            assets={"screen": Image(SCREEN)},
            questions={
                "pay_button": {"type": "ground", "instructions": "the 收付款 button"},
                "is_home": Noul(instructions="Is this the Alipay home screen?"),
            },
            extra_headers=IMG_HEADERS,
            response_model=GroundResponse,
        )
except TypeSafeInternalServerError as e:
    print("符合预期的 501：", e)


[pay_button] point = (400, 150) (0-999)  confidence = 0.813
        topk_points = [(400, 150)]
[scan_button] point = (95, 316) (0-999)  confidence = 0.349
        topk_points = [(95, 316), (117, 150)]
saved: ../assets/nb_out/ground_multi.png
model: gui-actor-3b | usage: {'input_tokens': 3185, 'output_tokens': 0} （每个不同指令一次 grounding pass，input_tokens 随之增加）
符合预期的 501： POST http://localhost:8008/v1/systemone: 501 question(s) is_home: the gui_actor backend does not answer ['noul'] questions yet (request_id=231d8c7746ba4c799104bb1ca00d75e8)


## §3 组合用例

- criteria 里可以放示例图（整张图作为选项、或 description + examples）
- 多个问题共享同一 state，一次请求并行回答
- Box 置信度低时：外扩 padding 再裁，用 Noul 验证目标是否完整落入裁剪

In [41]:
# Choice 的 criteria 三种形态：description+examples / 裸示例图 / 纯文本
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "top_area": Choice(
                instructions="What does the top area of `screen` contain?",
                criteria={
                    "icon_grid": {
                        "description": "A grid of four shortcut icons such as 扫一扫 and 收付款.",
                        "examples": [Image(GRID)],
                    },
                    "tab_bar": Image(TABBAR),  # 选项直接是一张示例图
                    "search_only": "Only a search bar and nothing else.",
                },
            )
        },
        extra_headers=IMG_HEADERS,
    )
show("top_area", r.answers["top_area"])

[top_area] choice = 'icon_grid'  confidence = 0.846
        probabilities = {'icon_grid': 0.846, 'tab_bar': 0.135, 'search_only': 0.018}


In [42]:
# Noul 的 criteria：true/false 各自可以用「描述 + 示例图」定义
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "has_grid": Noul(
                instructions="Does `screen` contain the shortcut icon grid shown in the example?",
                criteria={
                    "true": {
                        "description": "A 2x2 grid of shortcut icons including 扫一扫.",
                        "examples": [Image(GRID)],
                    },
                    "false": "No icon grid is visible.",
                },
            )
        },
        extra_headers=IMG_HEADERS,
    )
show("has_grid", r.answers["has_grid"])

[has_grid] noul = 0.331


In [43]:
# 多图 state：历史帧列表，问题引用数组路径（GUI agent 的「屏幕变了没」场景）
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={
            "frames": [
                {"seconds_ago": 2, "frame": Image(SCREEN)},
                {"seconds_ago": 0, "frame": Image(TABBAR)},
            ]
        },
        questions={
            "tabbar_changed": Noul(
                instructions="Did the bottom tab bar change between `frames`?"
            ),
            "new_content": Choice(
                instructions="What newly appeared in the latest frame?",
                criteria={
                    "dialog": "A dialog or popup covers part of the screen.",
                    "keyboard": "An on-screen keyboard appeared.",
                    "nothing": "Nothing new appeared.",
                },
            ),
        },
        extra_headers=IMG_HEADERS,
    )
for qid, ans in r.answers.items():
    show(qid, ans)

[tabbar_changed] noul = 0.028
[new_content] choice = 'nothing'  confidence = 0.901
        probabilities = {'dialog': 0.083, 'keyboard': 0.016, 'nothing': 0.901}


In [ ]:
# 单请求多问题：Noul 存在性 + Box 定位 + Choice 分类，共享同一 state 并行回答
with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={
            "pay_visible": Noul(instructions="Is there a 收付款 button in `screen`?"),
            "pay_button": Box(instructions="the 收付款 button in `screen`"),
            "selected_tab": Choice(
                instructions="Which bottom tab is currently selected in `screen`?",
                criteria={
                    "home": "首页 (Home) tab is highlighted.",
                    "life": "生活 (Life) tab is highlighted.",
                    "message": "消息 (Messages) tab is highlighted.",
                    "me": "我的 (Me) tab is highlighted.",
                },
            ),
        },
        extra_headers=IMG_HEADERS,
    )
for qid, ans in r.answers.items():
    show(qid, ans)

# 组合逻辑：Noul 说不存在就忽略 Box（Box 对不存在的目标也会返回一个框）
if r.answers["pay_visible"].noul < 0.5:
    print("pay_visible 低概率：忽略 box 答案")

In [ ]:
# Box 低置信度处理：低于阈值就外扩 padding 再裁，并用 Noul 验证目标完整落入裁剪
MIN_CONFIDENCE = 0.7  # 阈值需在自己的数据上调
PAD = 0.05           # 分数边外扩比例
MIN_SIDE = 300       # API 要求每边 >=256px；小元素裁剪时直接取稳妥下限

with client() as c:
    r = c.system_one(
        model=MODEL_PREVIEW,
        state={"screen": Image(SCREEN)},
        questions={"pay_button": Box(instructions="the 收付款 button in `screen`")},
        extra_headers=IMG_HEADERS,
    )

ans = r.answers["pay_button"]
show("pay_button", ans)
b = ans.box

if ans.confidence < MIN_CONFIDENCE:
    print(f"confidence {ans.confidence:.2f} < {MIN_CONFIDENCE}：外扩 {PAD} 后裁剪验证")
    # 1) 分数边外扩，夹在 [0, 1]
    edges = {
        "left": max(0.0, b.left - PAD),
        "top": max(0.0, b.top - PAD),
        "right": min(1.0, b.right + PAD),
        "bottom": min(1.0, b.bottom + PAD),
    }
    # 2) 换算成像素；小目标的裁剪很容易低于 API 的 256px/78400px 下限，
    #    以框中心为中心扩到至少 MIN_SIDExMIN_SIDE 再裁
    w, h = check_image(SCREEN)
    x0, y0 = round(edges["left"] * w), round(edges["top"] * h)
    x1, y1 = round(edges["right"] * w), round(edges["bottom"] * h)
    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
    half = MIN_SIDE // 2
    x0, x1 = max(0, cx - half), min(w, cx + half)
    y0, y1 = max(0, cy - half), min(h, cy + half)
    print(f"padded crop: ({x0}, {y0}) - ({x1}, {y1}) = {x1 - x0}x{y1 - y0}")

    crop_path = OUT / "pay_button_padded.png"
    PILImage.open(SCREEN).crop((x0, y0, x1, y1)).save(crop_path)
    cw, ch = check_image(crop_path)

    if cw >= 256 and ch >= 256 and cw * ch >= 78400:
        with client() as c:
            r2 = c.system_one(
                model=MODEL_PREVIEW,
                state={"crop": Image(crop_path)},
                questions={
                    "complete": Noul(
                        instructions="Is the whole 收付款 button inside `crop`, with no edge cutting it off?"
                    )
                },
                extra_headers=IMG_HEADERS,
            )
        show("complete", r2.answers["complete"])
    else:
        print("裁剪仍不满足 API 下限，跳过二次提问")
else:
    print(f"confidence {ans.confidence:.2f} >= {MIN_CONFIDENCE}：直接采用")

## §4 原生 HTTP：`assets` map 与内联引用

不用 SDK 时的请求形态：图片只在顶层 `assets` 出现一次，`state`/`instructions` 里用引用字符串指过去；
引用可以嵌在更长的话语中间（find-and-replace 组合式多模态）。这里用 `requests`，经同一代理出访。

In [45]:
import base64
import uuid

import requests


def asset_entry(path: Path) -> dict:
    fmt = "png" if path.suffix.lower() == ".png" else "jpg"
    return {
        "type": "image",
        "format": fmt,
        "data_base64": base64.b64encode(path.read_bytes()).decode("ascii"),
    }


screen_ref = str(uuid.uuid4())
grid_ref = str(uuid.uuid4())

body = {
    "model": MODEL_PREVIEW,
    "state": {"screen": screen_ref},
    "assets": {screen_ref: asset_entry(SCREEN), grid_ref: asset_entry(GRID)},
    "questions": {
        "inline_match": {
            "type": "noul",
            # 引用嵌在句子中间：模型在这里“看到”示例图
            "instructions": (
                f"Does the top of `screen` show the same icon layout as this example: {grid_ref}?"
            ),
        }
    },
}

resp = requests.post(
    os.environ['TYPESAFE_BASE_URL'] + "/v1/systemone",
    headers={
        "Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}",
        "TypeSafe-Danger-Zone-Images": "true",
    },
    json=body,
    timeout=120,
)
print(resp.status_code)
print(resp.json())

200
{'model': 'jev-preview', 'answers': {'inline_match': {'type': 'noul', 'noul': 0.86}}, 'usage': {'input_tokens': 1632, 'output_tokens': 0}, 'latency_ms': 558.0}


In [ ]:
# 错误用例：带图/Box 但不发 danger-zone 头 -> 400 Images are not turned on
from typesafe_sdk import TypeSafeBadRequestError

try:
    with client() as c:
        c.system_one(
            model=MODEL_PREVIEW,
            state={"screen": Image(SCREEN)},
            questions={"pay_button": Box(instructions="the 收付款 button in `screen`")},
            # 注意：没有 extra_headers=IMG_HEADERS
        )
except TypeSafeBadRequestError as e:
    print("符合预期的 400：", e)

## §5 模型、用量与备注

- `jev-latest` 当前仍是 1.13.0；`jev-preview` 是 1.14，**Box 必须用它**
- Ground 只有 `gui-actor-3b`（GUI-Actor pointer head）回答；clef 后端只答 noul/choice/score，两类不能混在一个请求
- 图片按 input token 计费（无单独图片费率），并占用上下文长度
- 长上下文头 `TypeSafe-Danger-Zone-Long-Context: true`（v14 实验特性，按需加）
- 更多模式（rerank、function calling、citation check 等）见 docs.typesafe.ai 的 cookbooks；本机已装 typesafe-ai skill 可随时查

In [47]:
with client() as c:
    # list() 返回可迭代对象，唯一元素是 ('models', (ModelMetadata, ...))
    _, metas = next(iter(c.models.list()))
    for m in metas:
        print(f"{m.name}: {m.description}（release: {m.release_date}）")

clef-flash: Clef joint-schema head on Qwen3.5, serving clef-flash（release: 2026-10-03）
gui-actor-3b: GUI-Actor pointer head on qwen2_5_vl, serving microsoft/GUI-Actor-3B-Qwen2.5-VL（release: 2026-10-08）
kev-latest: Clef joint-schema head on Qwen3.5, serving clef-flash（release: 2026-10-03）
jev-latest: Clef joint-schema head on Qwen3.5, serving clef-flash（release: 2026-10-03）
